```markdown
# Breast Cancer Classification Tutorial

In this notebook, we explore how to predict whether a tumor is **Malignant (0)** or **Benign (1)** using clinical data. We will compare basic Decision Trees, analyze the effect of tree depth, and use Random Forests with Grid Search optimization.
```

In [33]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split


from sklearn.datasets import load_breast_cancer

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


from sklearn.metrics import accuracy_score,precision_score,recall_score

from sklearn.metrics import confusion_matrix,classification_report

from sklearn.model_selection import GridSearchCV

```markdown
## 1. Data Loading and Splitting
First, we load the dataset and split it into training (80%) and testing (20%) sets. Using `stratify=y` ensures that the proportion of cancer cases is consistent in both sets.
```

In [34]:
df = load_breast_cancer(as_frame=True).frame
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,1.1760,1.2560,7.673,158.70,0.010300,0.02891,0.05198,0.02454,0.01114,0.004239,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,0.7655,2.4630,5.203,99.04,0.005769,0.02423,0.03950,0.01678,0.01898,0.002498,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,0.4564,1.0750,3.425,48.55,0.005903,0.03731,0.04730,0.01557,0.01318,0.003892,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,0.7260,1.5950,5.772,86.22,0.006522,0.06158,0.07117,0.01664,0.02324,0.006185,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [35]:
df.shape

(569, 31)

In [36]:
X = df.drop(['target'],axis=1)
y = df['target']
X_train , X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

```markdown
## 2. Baseline Model & Grid Search
We start with a default Decision Tree and then use `GridSearchCV` to automatically find the best settings for the tree's depth and splitting criteria.
```

In [37]:
model = DecisionTreeClassifier()

model.fit(X_train,y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.84      0.90      0.87        42
           1       0.94      0.90      0.92        72

    accuracy                           0.90       114
   macro avg       0.89      0.90      0.90       114
weighted avg       0.91      0.90      0.90       114



In [38]:
grid = {
    "criterion" : ['gini','entropy'],
    "max_depth" :[1,2,3,4,5,6,7,8,None],
}

grid_search = GridSearchCV(
    estimator = model,
    param_grid = grid,
    cv = 5
)

grid_search.fit(X_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'criterion': ['gini', 'entropy'],
                         'max_depth': [1, 2, 3, 4, 5, 6, 7, 8, None]})

In [39]:
y_pred = grid_search.predict(X_test)

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.83      0.95      0.89        42
           1       0.97      0.89      0.93        72

    accuracy                           0.91       114
   macro avg       0.90      0.92      0.91       114
weighted avg       0.92      0.91      0.91       114



# Training on different depth

```markdown
### Analyzing Model Depth (Overfitting vs. Underfitting)
As we increase `max_depth`, the model becomes more complex. Notice how the **Training Accuracy** usually goes up, but the **Testing Accuracy** might start to drop if the tree becomes too deep and starts 'memorizing' noise (overfitting).
```

In [40]:
model_1 = DecisionTreeClassifier(max_depth=2)
model_1.fit(X_train, y_train)
# training data accuracy

y_pred_train = model_1.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")

# testing data accuracy
y_pred = model_1.predict(X_test)
print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9582417582417583
Testing Accuracy: 0.8947368421052632


In [41]:
model_2 = DecisionTreeClassifier(max_depth=3)
model_2.fit(X_train, y_train)
# training data accuracy

y_pred_train = model_2.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")

# testing data accuracy
y_pred = model_2.predict(X_test)
print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9758241758241758
Testing Accuracy: 0.9385964912280702


In [42]:
model_3 = DecisionTreeClassifier(max_depth=4)
model_3.fit(X_train, y_train)
# training data accuracy

y_pred_train = model_3.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")

# testing data accuracy
y_pred = model_3.predict(X_test)
print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9868131868131869
Testing Accuracy: 0.9385964912280702


In [43]:
model_4 = DecisionTreeClassifier(max_depth=5)
model_4.fit(X_train, y_train)
# training data accuracy

y_pred_train = model_4.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")

# testing data accuracy
y_pred = model_4.predict(X_test)
print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9934065934065934
Testing Accuracy: 0.9210526315789473


In [44]:
#grid search
# training data accuracy
y_pred_train = grid_search.predict(X_train)
print(f"Training Accuracy: {accuracy_score(y_train,y_pred_train)}")


# testing accuracy
y_pred = grid_search.predict(X_test)

print(f"Testing Accuracy: {accuracy_score(y_test,y_pred)}")

Training Accuracy: 0.9934065934065934
Testing Accuracy: 0.9122807017543859
